# Coding Assignment 3: Image-Based Cancer Diagnosis Using Convolutional Neural Networks (CNNs)

# ID 2671508 - Tasnuba Tasnim

## Download the BUSI dataset from Kaggle

In [ ]:
!pip install -q kagglehub

from pathlib import Path
import kagglehub

KAGGLE_DATASET_HANDLE = (
    "subhajournal/busi-breast-ultrasound-images-dataset"
)

# Download the dataset
downloaded_dataset_path = Path(
    kagglehub.dataset_download(
        KAGGLE_DATASET_HANDLE
    )
)

print("Dataset downloaded successfully.")
print(f"Download location: {downloaded_dataset_path}")

# Locate the directory containing the BUSI class folders
required_folders = {
    "benign",
    "malignant",
    "normal"
}

def contains_class_folders(directory):
    if not directory.is_dir():
        return False

    child_folders = {
        child.name.lower()
        for child in directory.iterdir()
        if child.is_dir()
    }

    return required_folders.issubset(child_folders)

candidate_directories = []

if contains_class_folders(downloaded_dataset_path):
    candidate_directories.append(downloaded_dataset_path)

for directory in downloaded_dataset_path.rglob("*"):
    if contains_class_folders(directory):
        candidate_directories.append(directory)


if not candidate_directories:
    raise FileNotFoundError(
        "Could not find the benign, malignant and normal folders."
    )

# Used by Block 2
DATASET_ROOT = candidate_directories[0]

print(f"Dataset root: {DATASET_ROOT}")

## Phase A: Computer Vision Data Pipeline (PyTorch / TensorFlow)

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch

from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 1. Configuration

IMAGE_SIZE = 224
BATCH_SIZE = 32
RANDOM_STATE = 42
NUM_WORKERS = 0

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

COMPUTING_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Computing device: {COMPUTING_DEVICE}")

# 2. Collect image paths and binary labels

# Binary classification:
# 0 = Non-malignant: benign and normal
# 1 = Malignant

CLASS_MAPPING = {
    "benign": 0,
    "normal": 0,
    "malignant": 1
}

CLASS_NAMES = {
    0: "Non-malignant",
    1: "Malignant"
}

VALID_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff"
}

image_records = []

for class_folder, binary_label in CLASS_MAPPING.items():

    class_directory = Path(DATASET_ROOT) / class_folder

    if not class_directory.exists():
        raise FileNotFoundError(
            f"Class directory not found: {class_directory}"
        )

    for image_path in class_directory.rglob("*"):

        if not image_path.is_file():
            continue

        if image_path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        # Exclude BUSI segmentation masks
        if "_mask" in image_path.stem.lower():
            continue

        image_records.append(
            {
                "image_path": str(image_path),
                "label": binary_label
            }
        )

if not image_records:
    raise ValueError(
        "No valid classification images were found."
    )

all_labels = np.array(
    [record["label"] for record in image_records]
)

all_indices = np.arange(len(image_records))

print(f"Total classification images: {len(image_records)}")
print(
    f"Non-malignant images: "
    f"{np.sum(all_labels == 0)}"
)
print(
    f"Malignant images: "
    f"{np.sum(all_labels == 1)}"
)

# 3. Stratified 70% / 15% / 15% split
# First split:
# 70% training and 30% temporary

train_indices, temporary_indices = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=all_labels
)

# Second split:
# 15% validation and 15% testing

validation_indices, test_indices = train_test_split(
    temporary_indices,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=all_labels[temporary_indices]
)

train_records = [
    image_records[index]
    for index in train_indices
]

validation_records = [
    image_records[index]
    for index in validation_indices
]

test_records = [
    image_records[index]
    for index in test_indices
]

print("\nDataset split:")
print(f"Training:   {len(train_records)} images")
print(f"Validation: {len(validation_records)} images")
print(f"Testing:    {len(test_records)} images")

def print_class_balance(split_name, records):
    labels = np.array(
        [record["label"] for record in records]
    )

    print(
        f"{split_name}: "
        f"Non-malignant={np.sum(labels == 0)}, "
        f"Malignant={np.sum(labels == 1)}"
    )

print("\nClass balance after stratification:")
print_class_balance("Training", train_records)
print_class_balance("Validation", validation_records)
print_class_balance("Testing", test_records)

# 4. Custom PyTorch Dataset

class CancerImageDataset(Dataset):
    """
    Custom dataset for loading and transforming cancer images.
    """

    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):

        record = self.records[index]

        image_path = record["image_path"]
        binary_label = record["label"]

        with Image.open(image_path) as loaded_image:
            image = loaded_image.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label_tensor = torch.tensor(
            [binary_label],
            dtype=torch.float32
        )

        return image, label_tensor

# 5. Calculate dataset-wide mean and standard deviation

statistics_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor()
])


statistics_dataset = CancerImageDataset(
    records=image_records,
    transform=statistics_transform
)

statistics_loader = DataLoader(
    statistics_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

def calculate_dataset_statistics(data_loader):

    channel_sum = torch.zeros(
        3,
        dtype=torch.float64
    )

    channel_squared_sum = torch.zeros(
        3,
        dtype=torch.float64
    )

    total_pixels = 0

    for image_batch, _ in data_loader:

        image_batch = image_batch.to(
            dtype=torch.float64
        )

        batch_size, channels, height, width = (
            image_batch.shape
        )

        pixel_count = batch_size * height * width

        channel_sum += image_batch.sum(
            dim=(0, 2, 3)
        )

        channel_squared_sum += (
            image_batch ** 2
        ).sum(
            dim=(0, 2, 3)
        )

        total_pixels += pixel_count

    dataset_mean = channel_sum / total_pixels

    dataset_variance = (
        channel_squared_sum / total_pixels
    ) - dataset_mean.pow(2)

    dataset_variance = torch.clamp(
        dataset_variance,
        min=0
    )

    dataset_std = torch.sqrt(
        dataset_variance
    )

    return (
        dataset_mean.tolist(),
        dataset_std.tolist()
    )

dataset_mean, dataset_std = (
    calculate_dataset_statistics(
        statistics_loader
    )
)

dataset_std = [
    value if value > 0 else 1.0
    for value in dataset_std
]

print("\nDataset-wide RGB mean:")
print(dataset_mean)

print("\nDataset-wide RGB standard deviation:")
print(dataset_std)

# 6. Image augmentation and normalisation

# Augmentation is used only for training images.

training_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.3
    ),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=dataset_mean,
        std=dataset_std
    )
])

# Validation and testing use deterministic preprocessing.

evaluation_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=dataset_mean,
        std=dataset_std
    )
])

# 7. Create Dataset objects

train_dataset = CancerImageDataset(
    records=train_records,
    transform=training_transform
)

validation_dataset = CancerImageDataset(
    records=validation_records,
    transform=evaluation_transform
)

test_dataset = CancerImageDataset(
    records=test_records,
    transform=evaluation_transform
)

# 8. Create DataLoaders

train_data_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

validation_data_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_data_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# 9. Verify the completed pipeline

sample_images, sample_labels = next(
    iter(train_data_loader)
)

print("\nData pipeline completed.")
print(f"Training batches: {len(train_data_loader)}")
print(f"Validation batches: {len(validation_data_loader)}")
print(f"Testing batches: {len(test_data_loader)}")

print(
    f"Sample image batch shape: "
    f"{sample_images.shape}"
)

print(
    f"Sample label batch shape: "
    f"{sample_labels.shape}"
)

## Phase B: CNN Architecture & Training

### Custom CNN

In [ ]:
import copy

import torch
import torch.nn as nn
import torch.optim as optim

# 1. Custom CNN architecture

class CustomCancerCNN(nn.Module):
    """
    Custom convolutional neural network for binary cancer-image
    classification.
    """

    def __init__(self):
        super().__init__()

        # Feature extraction section:
        # Each convolutional layer is followed by:
        # Batch Normalisation -> ReLU -> Max Pooling

        self.feature_extractor = nn.Sequential(

            # Convolutional Block 1
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Convolutional Block 2
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Convolutional Block 3
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Reduce the final feature-map dimensions.
            nn.AdaptiveAvgPool2d((4, 4))
        )

        # Classification head:
        # Flatten -> Dense hidden layer -> Dropout ->
        # Single output node -> Sigmoid

        self.classification_head = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),
            nn.ReLU(),

            # Required regularisation range: 0.3–0.5
            nn.Dropout(p=0.4),

            nn.Linear(
                256,
                1
            ),
            nn.Sigmoid()
        )

    def forward(self, input_images):

        extracted_features = self.feature_extractor(
            input_images
        )

        output_probability = self.classification_head(
            extracted_features
        )

        return output_probability

# Initialize and transfer the model to the selected device.

cancer_cnn_model = CustomCancerCNN().to(
    COMPUTING_DEVICE
)

print(cancer_cnn_model)

### Loss function and optimiser

In [ ]:
# 2. Loss function and optimiser

LEARNING_RATE = 0.001

# Binary Cross-Entropy loss for binary classification.

binary_loss_function = nn.BCELoss()

# Adam optimiser.

model_optimizer = optim.Adam(
    cancer_cnn_model.parameters(),
    lr=LEARNING_RATE
)

print(f"Loss function: {binary_loss_function}")
print(f"Optimiser: {model_optimizer.__class__.__name__}")
print(f"Learning rate: {LEARNING_RATE}")

### Training function

In [ ]:
# 3. Metric-history containers

training_history = {
    "training_loss": [],
    "validation_loss": [],
    "training_accuracy": [],
    "validation_accuracy": []
}

# 4. Training function

def train_one_epoch(
    model,
    data_loader,
    loss_function,
    optimizer,
    device
):
    """
    Train the CNN for one epoch and return average loss
    and classification accuracy.
    """

    model.train()

    total_loss = 0.0
    total_correct_predictions = 0
    total_samples = 0

    for batch_images, batch_labels in data_loader:

        batch_images = batch_images.to(
            device,
            non_blocking=True
        )

        batch_labels = (
            batch_labels
            .float()
            .view(-1, 1)
            .to(
                device,
                non_blocking=True
            )
        )

        # Remove gradients from the previous iteration.
        optimizer.zero_grad(set_to_none=True)

        # Forward propagation.
        predicted_probabilities = model(
            batch_images
        )

        loss = loss_function(
            predicted_probabilities,
            batch_labels
        )

        # Backpropagation.
        loss.backward()

        # Update the model parameters.
        optimizer.step()

        current_batch_size = batch_images.size(0)

        total_loss += (
            loss.item()
            * current_batch_size
        )

        predicted_classes = (
            predicted_probabilities >= 0.5
        ).float()

        total_correct_predictions += (
            predicted_classes == batch_labels
        ).sum().item()

        total_samples += current_batch_size

    average_training_loss = (
        total_loss / total_samples
    )

    training_accuracy = (
        total_correct_predictions
        / total_samples
    )

    return (
        average_training_loss,
        training_accuracy
    )

### Validation function

In [ ]:
# 5. Validation function

def validate_one_epoch(
    model,
    data_loader,
    loss_function,
    device
):
    """
    Evaluate the CNN on the validation partition and return
    average loss and classification accuracy.
    """

    model.eval()

    total_loss = 0.0
    total_correct_predictions = 0
    total_samples = 0

    # Gradient computation is unnecessary during validation.
    with torch.no_grad():

        for batch_images, batch_labels in data_loader:

            batch_images = batch_images.to(
                device,
                non_blocking=True
            )

            batch_labels = (
                batch_labels
                .float()
                .view(-1, 1)
                .to(
                    device,
                    non_blocking=True
                )
            )

            predicted_probabilities = model(
                batch_images
            )

            loss = loss_function(
                predicted_probabilities,
                batch_labels
            )

            current_batch_size = batch_images.size(0)

            total_loss += (
                loss.item()
                * current_batch_size
            )

            predicted_classes = (
                predicted_probabilities >= 0.5
            ).float()

            total_correct_predictions += (
                predicted_classes == batch_labels
            ).sum().item()

            total_samples += current_batch_size

    average_validation_loss = (
        total_loss / total_samples
    )

    validation_accuracy = (
        total_correct_predictions
        / total_samples
    )

    return (
        average_validation_loss,
        validation_accuracy
    )

### Train the CNN for at least 20 epochs

In [ ]:
# 6. Train the CNN for at least 20 epochs

best_validation_loss = float("inf")
best_model_weights = None

print("\nStarting CNN training...\n")

NUM_EPOCHS = 20
for epoch_number in range(1, NUM_EPOCHS + 1):

    # Train for one epoch.
    training_loss, training_accuracy = (
        train_one_epoch(
            model=cancer_cnn_model,
            data_loader=train_data_loader,
            loss_function=binary_loss_function,
            optimizer=model_optimizer,
            device=COMPUTING_DEVICE
        )
    )

    # Evaluate on the validation set.
    validation_loss, validation_accuracy = (
        validate_one_epoch(
            model=cancer_cnn_model,
            data_loader=validation_data_loader,
            loss_function=binary_loss_function,
            device=COMPUTING_DEVICE
        )
    )

    # Store all mandatory epoch-level metrics.
    training_history["training_loss"].append(
        training_loss
    )

    training_history["validation_loss"].append(
        validation_loss
    )

    training_history["training_accuracy"].append(
        training_accuracy
    )

    training_history["validation_accuracy"].append(
        validation_accuracy
    )

    # Retain the model state with the lowest validation loss.
    if validation_loss < best_validation_loss:

        best_validation_loss = validation_loss

        best_model_weights = copy.deepcopy(
            cancer_cnn_model.state_dict()
        )

    print(
        f"Epoch {epoch_number:02d}/{NUM_EPOCHS} | "
        f"Training Loss: {training_loss:.4f} | "
        f"Validation Loss: {validation_loss:.4f} | "
        f"Training Accuracy: {training_accuracy:.4f} | "
        f"Validation Accuracy: {validation_accuracy:.4f}"
    )

# Restore the model with the best validation loss.

if best_model_weights is not None:

    cancer_cnn_model.load_state_dict(
        best_model_weights
    )


print("\nCNN training completed.")
print(
    f"Best validation loss: "
    f"{best_validation_loss:.4f}"
)
# 7. Verify that all metrics were tracked

print("\nNumber of recorded epochs:")

print(
    "Training loss:",
    len(training_history["training_loss"])
)

print(
    "Validation loss:",
    len(training_history["validation_loss"])
)

print(
    "Training accuracy:",
    len(training_history["training_accuracy"])
)

print(
    "Validation accuracy:",
    len(training_history["validation_accuracy"])
)

## Phase 3: Deep Learning Diagnostics & Visualization (Matplotlib)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# 1. Create a folder for the saved figures

PLOT_DIRECTORY = Path("phase_c_plots")
PLOT_DIRECTORY.mkdir(exist_ok=True)

# 2. Learning curves

epochs = range(
    1,
    len(training_history["training_loss"]) + 1
)


figure, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

# Training versus validation loss

axes[0].plot(
    epochs,
    training_history["training_loss"],
    marker="o",
    label="Training Loss"
)

axes[0].plot(
    epochs,
    training_history["validation_loss"],
    marker="o",
    label="Validation Loss"
)

axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.6)

# Training versus validation accuracy

axes[1].plot(
    epochs,
    training_history["training_accuracy"],
    marker="o",
    label="Training Accuracy"
)

axes[1].plot(
    epochs,
    training_history["validation_accuracy"],
    marker="o",
    label="Validation Accuracy"
)

axes[1].set_title("Training and Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.6)


plt.tight_layout()

plt.savefig(
    PLOT_DIRECTORY / "cnn_learning_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 3. Generate predictions on the unseen testing partition

cancer_cnn_model.eval()

test_true_labels = []
test_probabilities = []


with torch.no_grad():

    for batch_images, batch_labels in test_data_loader:

        batch_images = batch_images.to(
            COMPUTING_DEVICE
        )

        predicted_probabilities = cancer_cnn_model(
            batch_images
        )

        test_true_labels.extend(
            batch_labels
            .cpu()
            .numpy()
            .ravel()
            .tolist()
        )

        test_probabilities.extend(
            predicted_probabilities
            .cpu()
            .numpy()
            .ravel()
            .tolist()
        )


test_true_labels = np.array(
    test_true_labels,
    dtype=int
)

test_probabilities = np.array(
    test_probabilities
)


# Convert probabilities into binary predictions.

test_predicted_classes = (
    test_probabilities >= 0.5
).astype(int)


# Calculate final test accuracy.

test_accuracy = np.mean(
    test_predicted_classes == test_true_labels
)

print(f"Final test accuracy: {test_accuracy:.4f}")



# 4. ROC curve and AUC score

false_positive_rate, true_positive_rate, thresholds = (
    roc_curve(
        test_true_labels,
        test_probabilities
    )
)

test_auc = roc_auc_score(
    test_true_labels,
    test_probabilities
)


plt.figure(figsize=(8, 6))

plt.plot(
    false_positive_rate,
    true_positive_rate,
    linewidth=2,
    label=f"Custom CNN — AUC = {test_auc:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.title("ROC Curve on the Unseen Testing Partition")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()

plt.savefig(
    PLOT_DIRECTORY / "cnn_test_roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Final test ROC-AUC score: {test_auc:.4f}")

# 5. Test-set confusion matrix

confusion_matrix_values = confusion_matrix(
    test_true_labels,
    test_predicted_classes,
    labels=[0, 1]
)

true_negative, false_positive, false_negative, true_positive = (
    confusion_matrix_values.ravel()
)


confusion_display = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_values,
    display_labels=[
        "Non-malignant",
        "Malignant"
    ]
)

confusion_display.plot(
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title("Confusion Matrix on the Testing Partition")
plt.tight_layout()

plt.savefig(
    PLOT_DIRECTORY / "cnn_test_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 6. Display confusion-matrix results

print("\nConfusion matrix results:")

print(f"True Negatives:  {true_negative}")
print(f"False Positives: {false_positive}")
print(f"False Negatives: {false_negative}")
print(f"True Positives:  {true_positive}")

print(
    "\nFalse Positive: A non-malignant image was "
    "predicted as malignant."
)

print(
    "False Negative: A malignant image was "
    "predicted as non-malignant."
)
